[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshitaBhatt486/ML-and-data/blob/main/DOM-project/notebooks/02_Data_Understanding.ipynb)

# Phase 2 – Data Understanding
**Dataset:** Tourism & Hospitality Industry Analysis Dataset (Kaggle, `smithmurphy`)

This notebook:
1. Loads the raw dataset (works both in **Google Colab** and locally inside the repo).
2. Describes the data source and dataset.
3. Builds the **metadata table** with the **measurement scale** (Nominal / Ordinal / Interval / Ratio) of every variable.
4. Profiles the variables and checks data quality.
5. Summarises the implications for the following phases.

*Run all cells top to bottom (`Runtime → Run all` in Colab).*

## 0. Setup

In [ ]:
import os
import urllib.request
import zipfile
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
print("Running in Colab:", IN_COLAB)

### Load the raw dataset
The loader tries, in order:
1. the copy in this repo (`../data/raw/` when the notebook is opened locally),
2. the copy on GitHub (when opened in Colab from the repo),
3. Kaggle's public download endpoint,
4. a manual file upload (Colab only) as a last resort.

The file is saved to `data/raw/` so later notebooks can reuse it.

In [ ]:
FILE = "Tourism_Hospitality_Industry_Analysis.csv"
GITHUB_RAW = "https://raw.githubusercontent.com/IshitaBhatt486/ML-and-data/main/DOM-project/data/raw/" + FILE
KAGGLE_ZIP = "https://www.kaggle.com/api/v1/datasets/download/smithmurphy/tourism-and-hospitality-industry-analysis-dataset"

# Project root: the repo folder when run locally, the Colab working dir otherwise
PROJECT = ".." if os.path.exists(os.path.join("..", "data", "raw", FILE)) else "."
RAW_DIR = os.path.join(PROJECT, "data", "raw")
OUT_DIR = os.path.join(PROJECT, "outputs", "phase2")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
RAW_PATH = os.path.join(RAW_DIR, FILE)

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "python-notebook"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if not os.path.exists(RAW_PATH):
    try:
        data = fetch(GITHUB_RAW)
        open(RAW_PATH, "wb").write(data)
        print("Downloaded from GitHub")
    except Exception as e:
        print("GitHub copy not available:", e)
        try:
            with zipfile.ZipFile(io.BytesIO(fetch(KAGGLE_ZIP))) as z:
                open(RAW_PATH, "wb").write(z.read(FILE))
            print("Downloaded from Kaggle")
        except Exception as e2:
            print("Kaggle download failed:", e2)
            if IN_COLAB:
                from google.colab import files
                print("Please upload", FILE)
                up = files.upload()
                open(RAW_PATH, "wb").write(up[FILE])

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, df.shape)

## 1. Data source

| Item | Detail |
|---|---|
| Dataset | Tourism & Hospitality Industry Analysis Dataset |
| Publisher | Kaggle user `smithmurphy` |
| URL | https://www.kaggle.com/datasets/smithmurphy/tourism-and-hospitality-industry-analysis-dataset |
| File | `Tourism_Hospitality_Industry_Analysis.csv` (73.6 KB; file timestamp 13 Mar 2025) |
| Type | Public secondary dataset (Kaggle) |
| Access | Public download; no login, scraping or paywall bypass |
| Privacy | No PII or PHI. Records are aggregated at the city and month level, with no names, emails, phone numbers or individual tourists. |

### Data sourcing log (for the report appendix)

| Source URL | Date of collection | Tool / method |
|---|---|---|
| https://www.kaggle.com/datasets/smithmurphy/tourism-and-hospitality-industry-analysis-dataset | 9 November 2026 | Direct download of the public ZIP through Kaggle's dataset download endpoint, then unzipped. The file was not modified. |

## 2. Dataset description

In [ ]:
print(f"Observations (rows): {df.shape[0]}")
print(f"Variables (columns): {df.shape[1]}")
print(f"Meets minimum (>=500 rows, >=8 variables): {df.shape[0] >= 500 and df.shape[1] >= 8}")
print(f"Countries: {df['Country'].nunique()}, Cities: {df['City'].nunique()}")
print(f"Years: {sorted(df['Year'].unique().tolist())}, Months: {df['Month'].nunique()}")
df.head()

In [ ]:
df.info()

In [ ]:
# Cities covered in each country
df.groupby("Country")["City"].unique().apply(lambda c: ", ".join(sorted(c))).to_frame("Cities")

**Unit of observation:** one record for one **city in one month** (and year), with a purpose-of-visit segment and a hotel-rating tier.

| Group | Variables | Business question |
|---|---|---|
| Location and time | Country, City, Year, Month | Q1, Q4 |
| Demand | Number_of_Tourists, Purpose_of_Visit, Average_Length_of_Stay | Q1, Q2, Q4 |
| Spending and economic value | Tourist_Spending_USD, Tourism_Revenue_USD, Employment_in_Tourism, Contribution_to_GDP_Percent | Q2 |
| Hospitality capacity | Number_of_Hotels, Hotel_Occupancy_Rate, Hotel_Rating, Average_Room_Price_USD | Q3 |
| Accessibility | Number_of_Flights, Airport_Passenger_Traffic, Transport_Infrastructure_Quality | Q4 |
| Sustainability | Eco_Tourism_Revenue_USD, Carbon_Footprint_kg, Waste_Management_Rating | Supporting |
| Experience and popularity | Tourist_Satisfaction_Score, Number_of_Online_Reviews | Q4 |

## 3. Metadata table (with measurement scale)

In [ ]:
META = [
    # Variable, Description, Unit, Measurement scale, Data type, Role, Business question
    ("Country", "Destination country", "-", "Nominal", "Categorical", "Dimension (geography)", "Q1-Q4"),
    ("City", "Destination city", "-", "Nominal", "Categorical", "Dimension (geography)", "Q1-Q3"),
    ("Year", "Calendar year of the record", "Year", "Interval", "Discrete numeric", "Dimension (time)", "Q1, Q4"),
    ("Month", "Calendar month (1 = Jan ... 12 = Dec)", "Month", "Ordinal", "Discrete (coded)", "Dimension (seasonality)", "Q1, Q4"),
    ("Number_of_Tourists", "Tourist arrivals in the city for the month", "Persons", "Ratio", "Discrete numeric", "Target (Phase 6)", "Q1, Q3, Q4"),
    ("Purpose_of_Visit", "Main purpose of travel", "-", "Nominal", "Categorical", "Segment", "Q2, Q4"),
    ("Average_Length_of_Stay", "Average nights per tourist", "Nights", "Ratio", "Continuous", "Predictor / KPI", "Q2, Q3"),
    ("Tourist_Spending_USD", "Average spend per tourist", "USD", "Ratio", "Continuous", "Predictor / KPI", "Q2"),
    ("Hotel_Occupancy_Rate", "Share of rooms occupied", "%", "Ratio", "Continuous", "Capacity-utilisation KPI", "Q3"),
    ("Number_of_Hotels", "Hotels operating in the city", "Count", "Ratio", "Discrete numeric", "Capacity predictor", "Q3, Q4"),
    ("Hotel_Rating", "Hotel star tier (Unrated, 1-5 star)", "Stars", "Ordinal", "Categorical (ordered)", "Quality tier", "Q2, Q3"),
    ("Average_Room_Price_USD", "Average nightly room rate", "USD / night", "Ratio", "Continuous", "Pricing KPI", "Q2, Q3"),
    ("Tourism_Revenue_USD", "Tourism revenue (= Tourists x Spending / 1000)", "USD '000 in effect", "Ratio", "Continuous", "Economic KPI (derived)", "Q2"),
    ("Employment_in_Tourism", "People employed in tourism", "Persons", "Ratio", "Discrete numeric", "Economic impact", "Q2, Q4"),
    ("Contribution_to_GDP_Percent", "Tourism share of GDP", "%", "Ratio", "Continuous", "Economic KPI", "Q2"),
    ("Number_of_Flights", "Incoming flights", "Count", "Ratio", "Discrete numeric", "Accessibility predictor", "Q4"),
    ("Airport_Passenger_Traffic", "Airport passengers handled", "Persons", "Ratio", "Discrete numeric", "Accessibility predictor", "Q4"),
    ("Transport_Infrastructure_Quality", "Transport quality index", "Score 1-10", "Ordinal", "Index score", "Accessibility predictor", "Q4"),
    ("Eco_Tourism_Revenue_USD", "Eco-tourism revenue (5-30% of tourism revenue)", "USD '000 in effect", "Ratio", "Continuous", "Sustainability KPI (derived)", "Q2"),
    ("Carbon_Footprint_kg", "Tourism carbon emissions (100-500 kg per tourist)", "kg CO2", "Ratio", "Continuous", "Sustainability KPI (derived)", "Supporting"),
    ("Waste_Management_Rating", "Waste-management rating", "Score 1-10", "Ordinal", "Index score", "Sustainability indicator", "Supporting"),
    ("Tourist_Satisfaction_Score", "Average tourist satisfaction", "Score 5-10", "Ordinal", "Index score", "Experience KPI / predictor", "Q4"),
    ("Number_of_Online_Reviews", "Online reviews of the destination", "Count", "Ratio", "Discrete numeric", "Popularity predictor", "Q4"),
]
meta = pd.DataFrame(META, columns=["Variable", "Description", "Unit", "Measurement_Scale",
                                   "Data_Type", "Analytical_Role", "Business_Question"])
assert set(meta["Variable"]) == set(df.columns), "metadata out of sync with dataset"

HOTEL_RATING_ORDER = ["Unrated", "1-star", "2-star", "3-star", "4-star", "5-star"]

def observed(col):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return f"{s.min():,g} to {s.max():,g}"
    levels = HOTEL_RATING_ORDER if col == "Hotel_Rating" else sorted(s.unique())
    return f"{len(levels)} levels: " + ", ".join(levels) if len(levels) <= 10 else f"{len(levels)} levels"

meta["Pandas_dtype"] = meta["Variable"].map(lambda c: str(df[c].dtype))
meta["Observed_Range_or_Levels"] = meta["Variable"].map(observed)
meta["Distinct_Values"] = meta["Variable"].map(lambda c: df[c].nunique())
meta["Missing"] = meta["Variable"].map(lambda c: df[c].isna().sum())
meta.index = range(1, len(meta) + 1)
meta.to_csv(os.path.join(OUT_DIR, "metadata_table.csv"), index=False)
meta

In [ ]:
# Measurement scale summary
meta.groupby("Measurement_Scale")["Variable"].agg(Count="count", Variables=lambda v: ", ".join(v)) \
    .reindex(["Nominal", "Ordinal", "Interval", "Ratio"])

### Justification of the scale choices that can be argued
- **Year: interval.** Differences are meaningful (2024 − 2020 = 4 years), but year 0 is not a true zero, so ratios between years mean nothing.
- **Month: ordinal.** Months have a natural order, but the scale is cyclical: December is next to January, not 11 units away. For modelling, treat Month as a category or as a season, not as a linear number.
- **Hotel_Rating: ordinal.** 1-star < … < 5-star is ordered, but the gaps between tiers are not equal. "Unrated" is a missing classification rather than "0 stars", so it is kept as a separate level.
- **Score variables (Transport infrastructure, Waste management, Satisfaction): ordinal.** These are subjective index scores, so a rise from 4 to 5 is not guaranteed to equal a rise from 8 to 9. By convention they may be *treated* as approximately interval when computing means (Phase 4), and we state that assumption.
- **Percentages (occupancy, GDP share): ratio.** 0 % is a true zero, and 80 % occupancy is twice 40 %.
- **Counts and money: ratio.** They have a true zero and ratios are meaningful.

## 4. First look at the variables
### Numeric variables

In [ ]:
num_cols = df.select_dtypes("number").columns
cat_cols = df.select_dtypes(exclude="number").columns

profile = df[num_cols].describe().T
profile["CV_%"] = profile["std"] / profile["mean"] * 100
profile["Skewness"] = df[num_cols].skew()
profile["Excess_Kurtosis"] = df[num_cols].kurt()
profile.to_csv(os.path.join(OUT_DIR, "numeric_profile.csv"))
profile

### Categorical variables

In [ ]:
cat_profile = []
for c in cat_cols:
    vc = df[c].value_counts()
    if c == "Hotel_Rating":
        vc = vc.reindex(HOTEL_RATING_ORDER)
    cat_profile.append(pd.DataFrame({"Variable": c, "Level": vc.index, "Count": vc.values,
                                     "Percent": (vc.values / len(df) * 100).round(1)}))
cat_profile = pd.concat(cat_profile, ignore_index=True)
cat_profile.to_csv(os.path.join(OUT_DIR, "categorical_profile.csv"), index=False)

for c in ["Purpose_of_Visit", "Hotel_Rating"]:
    display(cat_profile[cat_profile["Variable"] == c])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, c in zip(axes.flat, ["Country", "Purpose_of_Visit", "Hotel_Rating", "Year"]):
    vc = df[c].value_counts()
    if c == "Hotel_Rating":
        vc = vc.reindex(HOTEL_RATING_ORDER)
    elif c == "Year":
        vc = vc.sort_index()
    vc = vc.iloc[::-1]
    ax.barh(vc.index.astype(str), vc.values, color="#2a6f97")
    ax.set_title(f"Records by {c.replace('_', ' ')}")
    ax.set_xlabel("Number of records")
    ax.tick_params(labelsize=8)
fig.suptitle("Phase 2 - Coverage of categorical / time variables", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(os.path.join(OUT_DIR, "fig_categorical_counts.png"), dpi=130)
plt.show()

### Distributions of numeric variables

In [ ]:
ncols = 4
nrows = int(np.ceil(len(num_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.0 * nrows))
for ax, c in zip(axes.flat, num_cols):
    if c in ("Year", "Month"):
        vc = df[c].value_counts().sort_index()
        ax.bar(vc.index.astype(str), vc.values, color="#2a6f97")
    else:
        ax.hist(df[c], bins=20, color="#2a6f97", edgecolor="white")
    ax.set_title(c.replace("_", " "), fontsize=9)
    ax.set_ylabel("Records", fontsize=7)
    ax.tick_params(labelsize=7)
for ax in list(axes.flat)[len(num_cols):]:
    ax.axis("off")
fig.suptitle(f"Phase 2 - Distribution of numeric variables (n = {len(df)})", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(os.path.join(OUT_DIR, "fig_distributions.png"), dpi=130)
plt.show()

**Observation:** almost every numeric variable is flat (skewness ≈ 0, excess kurtosis ≈ −1.2), i.e. **uniformly distributed** between a fixed minimum and maximum. Only Tourism_Revenue, Eco_Tourism_Revenue and Carbon_Footprint are right-skewed.

## 5. Data quality assessment

In [ ]:
print("Missing cells       :", int(df.isna().sum().sum()))
print("Duplicate rows      :", int(df.duplicated().sum()))
print("Cities in >1 country:", int((df.groupby("City")["Country"].nunique() > 1).sum()))

key = df.groupby(["City", "Year", "Month"]).size()
print(f"Repeated (City, Year, Month) keys: {(key > 1).sum()} keys covering {key[key > 1].sum()} rows")
print(f"Panel coverage: {len(df)} of {df['City'].nunique() * 60:,} possible city-month cells")
print("Cities with >1 Hotel_Rating:", int((df.groupby("City")["Hotel_Rating"].nunique() > 1).sum()),
      "of", df["City"].nunique())

### Are some columns calculated from others?

In [ ]:
rev_check = (df["Tourism_Revenue_USD"] - df["Number_of_Tourists"] * df["Tourist_Spending_USD"] / 1000).abs()
eco_share = df["Eco_Tourism_Revenue_USD"] / df["Tourism_Revenue_USD"]
co2_per_tourist = df["Carbon_Footprint_kg"] / df["Number_of_Tourists"]

print(f"max |Revenue - Tourists x Spending / 1000| = {rev_check.max():.4f}")
print(f"Eco-tourism revenue as share of revenue   : {eco_share.min():.3f} to {eco_share.max():.3f}")
print(f"Carbon footprint per tourist (kg)         : {co2_per_tourist.min():.1f} to {co2_per_tourist.max():.1f}")
print()
print("Rows where airport passengers < tourists :", int((df["Airport_Passenger_Traffic"] < df["Number_of_Tourists"]).sum()))
print(f"corr(Number_of_Flights, Airport_Passenger_Traffic) = "
      f"{df['Number_of_Flights'].corr(df['Airport_Passenger_Traffic']):+.3f}")

### Correlation between numeric variables

In [ ]:
corr = df[num_cols].corr()
corr.round(3).to_csv(os.path.join(OUT_DIR, "correlation_matrix.csv"))

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
labels = [c.replace("_", " ") for c in num_cols]
ax.set_xticks(range(len(num_cols)), labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(num_cols)), labels, fontsize=8)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        if i != j and abs(corr.iloc[i, j]) >= 0.2:
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Phase 2 - Correlation between numeric variables (|r| >= 0.20 labelled)")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_correlation_heatmap.png"), dpi=130)
plt.show()

### Association with the planned Phase 6 target (`Number_of_Tourists`)

In [ ]:
target = "Number_of_Tourists"
r_target = corr[target].drop(target).sort_values(key=abs, ascending=False).to_frame("Pearson_r")
r_target["Derived_from_target"] = r_target.index.isin(
    ["Tourism_Revenue_USD", "Eco_Tourism_Revenue_USD", "Carbon_Footprint_kg"])
r_target

In [ ]:
# One-way ANOVA: do tourist arrivals differ between groups?
anova = {}
for c in ["Country", "City", "Purpose_of_Visit", "Hotel_Rating", "Year", "Month"]:
    groups = [g[target].values for _, g in df.groupby(c) if len(g) > 1]
    anova[c] = stats.f_oneway(*groups).pvalue
pd.Series(anova, name="ANOVA p-value").to_frame()

### Summary of data quality

| Check | Result | Consequence for Phase 3 |
|---|---|---|
| Missing values | 0 cells | No imputation needed |
| Duplicate rows | 0 | No rows to remove |
| City–country mapping | Each city maps to exactly one country | Consistent |
| Data types | All numeric columns parse as numbers | Convert Month and Hotel_Rating to ordered categories, and Year and Month to a date |
| Repeated (City, Year, Month) keys | 30 keys repeat, covering 61 rows | A row is a segment record, not a unique panel cell. Do **not** treat these rows as duplicates. |
| Panel coverage | 500 of 3,360 possible city-month cells | The time series per city are sparse, so seasonality analysis should aggregate across cities and years |
| **Tourism_Revenue_USD is derived** | = Tourists × Spending ÷ 1000, exactly | Effectively in **thousand USD**; relabel it in Phase 3 |
| **Eco_Tourism_Revenue_USD is derived** | Always 5–30 % of revenue | Use it as a share of revenue |
| **Carbon_Footprint_kg is derived** | Always 100–500 kg × tourists | Report it per tourist |
| Airport passengers vs tourists | 36 rows have fewer passengers than tourists; flights vs passengers r ≈ −0.03 | Treat the accessibility variables with caution |
| Distributions | Nearly all uniform | Few or no statistical outliers expected in Phase 3 |

## 6. Important finding: the dataset is synthetic

The evidence above indicates this is a **simulated (synthetic) dataset**, not real tourism statistics:
1. Almost every numeric variable is uniformly distributed between round bounds (e.g. tourists 50k–200k, occupancy 40–90 %).
2. Variables are statistically independent. The only correlations above |0.2| are among columns calculated from tourists and spending.
3. Values that should be linked in reality are not, for example flights and airport passengers.

**Implications:**
- **Phases 3–5 can go ahead as planned.** Cleaning, descriptive statistics, visualisations and the Tableau dashboard all work on this data.
- **Phase 6 (predicting tourist arrivals) will probably give R² close to 0** when only legitimate predictors are used. Revenue, eco revenue and carbon footprint must be excluded, because they are calculated from the target (leakage). A near-zero R² is a valid, reportable result.
- The synthetic nature must be stated as a **limitation** in the report. Phase 7 recommendations should be framed as a demonstration of the method.
- Alternatives:
  - (a) Predict a target with a structural relationship, e.g. revenue from tourists and spending (a weaker business story).
  - (b) Switch to real open data before Phase 3.

## 7. Phase 2 summary
- The data comes from a public Kaggle dataset with 500 observations and 23 variables, covering 20 countries and 56 cities over 2020–2024. It meets the 500 × 8 minimum.
- Measurement scales: 3 nominal, 5 ordinal, 1 interval and 14 ratio.
- The data is complete: no missing values and no duplicate rows. The main Phase 3 tasks are type conversions, relabelling the revenue unit, and handling the derived columns.
- Three columns are derived from tourist arrivals, and the data appears synthetic, which limits what the Phase 6 model can find.

Outputs saved to `outputs/phase2/`.

➡️ **Next:** `03_Data_Preprocessing.ipynb`

In [ ]:
# List the files this notebook produced
sorted(os.listdir(OUT_DIR))